In [1]:
import os
import re
import json
import pandas as pd
from tqdm import tqdm


# utils

In [2]:
def read_jsonl(file_path):
    data = []
    with open(file_path, 'r') as json_file:
        for line in json_file:
            data.append(json.loads(line))
    return data


def extract_python_method_code(start_code: str, generated_code: str):
        
    # Find the start index of start_code in generated_code
    start_index = generated_code.find(start_code)
    if start_index == -1:
        print("Not Found")
        return generated_code, '', ''

    main_pattern = re.compile(r'if\s+__name__\s*==\s*[\'"]__main__[\'"]:\s*\n.*', re.DOTALL)
    function_pattern = re.compile(
        rf'({re.escape(start_code)}(?:[\s\S]*?)(?=\n(?!\s)|\Z))',
        re.MULTILINE | re.DOTALL
    )
    main_method = re.compile(r'def\s+main\s*\(.*?\)\s*:\s*\n(.*?)(?=\ndef\s+\w+\(|if\s+__name__|^\S|\Z)', re.DOTALL | re.MULTILINE)
    
    function_match = function_pattern.search(generated_code, start_index)
    if not function_match:
        print("Not Found")
        return generated_code[start_index:].strip(), '', ''
    
    method_code = function_match.group(1).strip()
    main_method_match = main_method.search(generated_code)
    main_match = main_pattern.search(generated_code)
    main_block = ''
    main_method_block = ''
    if main_match:
        main_block = main_match.group(0)
    if main_method_match:
        main_method_block = main_method_match.group(0)
        
    return method_code, main_method_block, main_block

# this will be used for gpt models because they are not completion models they are basically chat models so we cant ensure that we will have the code starting with same prompt
def extract_method_code_gpt(code, prompt):
    if not code:
        return prompt
    main_block_pattern = r'if\s+__name__\s*==\s*["\']__main__["\']\s*:.*'
    main_block_match = re.search(main_block_pattern, code, flags=re.MULTILINE)
    if main_block_match:
        # Remove everything from `if __name__ == "__main__":` onwards
        code = code[:main_block_match.start()]

    # Step 2: Keep only lines that start with `def`, `import`, or are indented (start with spaces or tabs)
    cleaned_lines = []
    for line in code.splitlines():
        stripped_line = line.strip()
        if stripped_line.startswith(('def ', 'from ', 'import ')) or line.startswith((' ', '\t')):
            cleaned_lines.append(line)

    # Step 3: Join the cleaned lines back into a single string
    cleaned_code = "\n".join(cleaned_lines)
    if not cleaned_code:
        return code

    return cleaned_code.strip()

In [3]:
# HUMAN_EVAL_PLUS = "../../data/humaneval_plus/HumanEvalPlus.jsonl"
# HUMAN_EVAL_PLUS = read_jsonl(HUMAN_EVAL_PLUS)

# merge humanEval Plus

## deepseek-coder

In [10]:
deepseek_coder = "../results/humaneval_plus/deepseek_coder"


In [11]:
base_codes = []
for exp in os.listdir(os.path.join(deepseek_coder, 'base')):
    if exp == "codes":
        pass
    else:
        res_file = os.path.join(deepseek_coder, 'base', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 164
        base_codes.append(data)
len(base_codes)

10

In [12]:
finetune_codes = []
for exp in os.listdir(os.path.join(deepseek_coder, 'finetuned')):
    if exp != "codes":
        res_file = os.path.join(deepseek_coder, 'finetuned', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 164
        finetune_codes.append(data)
len(finetune_codes)

10

In [13]:
merged_codes = []
for exp in os.listdir(os.path.join(deepseek_coder, 'merged')):
    if exp != "codes":
        res_file = os.path.join(deepseek_coder, 'merged', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 164
        merged_codes.append(data)
len(merged_codes)

10

In [14]:
merged_modelname = merged_codes[0][0]["result"][0]["model"].replace("/", "_")
finetune_model_name = finetune_codes[0][0]["result"][0]["model"].replace("/", "_")
base_modelname = base_codes[0][0]["result"][0]["model"].replace("/", "_")

In [21]:
m_task_id = []
m_prompt = []
m_codes = {}
for i in range(len(merged_codes)):
    codes = merged_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            m_task_id.append(codes[j]["task_id"])
            m_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    m_codes[f"exp_{i}"] = g_codes
m_codes["task_id"] = m_task_id
m_codes["prompt"] = m_prompt

df = pd.DataFrame(m_codes)
df.to_csv(os.path.join("merged_humaneval_plus", f"{merged_modelname}.csv"))


In [23]:
f_task_id = []
f_prompt = []
f_codes = {}
for i in range(len(finetune_codes)):
    codes = finetune_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            f_task_id.append(codes[j]["task_id"])
            f_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    f_codes[f"exp_{i}"] = g_codes
f_codes["task_id"] = f_task_id
f_codes["prompt"] = f_prompt

df = pd.DataFrame(f_codes)
df.to_csv(os.path.join("merged_humaneval_plus", f"{finetune_model_name}.csv"))

In [24]:
b_task_id = []
b_prompt = []
b_codes = {}
for i in range(len(base_codes)):
    codes = base_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            b_task_id.append(codes[j]["task_id"])
            b_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    b_codes[f"exp_{i}"] = g_codes
b_codes["task_id"] = b_task_id
b_codes["prompt"] = b_prompt

df = pd.DataFrame(b_codes)
df.to_csv(os.path.join("merged_humaneval_plus", f"{base_modelname}.csv"))

## codellama

In [25]:
codellama = "../results/humaneval_plus/codellama"


In [26]:
base_codes = []
for exp in os.listdir(os.path.join(codellama, 'base')):
    if exp!="codes":
        res_file = os.path.join(codellama, 'base', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 164
        base_codes.append(data)
len(base_codes)

10

In [27]:
finetune_codes = []

for exp in os.listdir(os.path.join(codellama, 'finetuned')):
    if exp != "codes":
        res_file = os.path.join(codellama, 'finetuned', exp, 'result.json')
        data = read_jsonl(res_file)
        assert (len(data)) == 164
        finetune_codes.append(data)

In [29]:
merged_codes = []
for exp in os.listdir(os.path.join(codellama, 'merged')):
    if exp!="codes":
        res_file = os.path.join(codellama, 'merged', exp, 'result.json')
        data = read_jsonl(res_file)
        assert (len(data)) == 164
        merged_codes.append(data)


In [30]:
merged_modelname = merged_codes[0][0]["result"][0]["model"].replace("/", "_")
finetune_model_name = finetune_codes[0][0]["result"][0]["model"].replace("/", "_")
base_modelname = base_codes[0][0]["result"][0]["model"].replace("/", "_")

In [34]:
m_task_id = []
m_prompt = []
m_codes = {}
for i in range(len(merged_codes)):
    codes = merged_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            m_task_id.append(codes[j]["task_id"])
            m_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    m_codes[f"exp_{i}"] = g_codes
m_codes["task_id"] = m_task_id
m_codes["prompt"] = m_prompt

df = pd.DataFrame(m_codes)
df.to_csv(os.path.join("merged_humaneval_plus", f"{merged_modelname}.csv"))

In [35]:
f_task_id = []
f_prompt = []
f_codes = {}
for i in range(len(finetune_codes)):
    codes = finetune_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            f_task_id.append(codes[j]["task_id"])
            f_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    f_codes[f"exp_{i}"] = g_codes
f_codes["task_id"] = f_task_id
f_codes["prompt"] = f_prompt

df = pd.DataFrame(f_codes)
df.to_csv(os.path.join("merged_humaneval_plus", f"{finetune_model_name}.csv"))

In [23]:
b_task_id = []
b_prompt = []
b_codes = {}
base_modelname = base_codes[0][0]["result"][0]["model"].replace("/", "_")
for i in range(len(base_codes)):
    codes = base_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            b_task_id.append(codes[j]["task_id"])
            b_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    b_codes[f"exp_{i}"] = g_codes
b_codes["task_id"] = b_task_id
b_codes["prompt"] = b_prompt

df = pd.DataFrame(b_codes)
df.to_csv(os.path.join("merged_humaneval_plus", f"{base_modelname}.csv"))

## GPT

In [3]:
gpt_results = "../results/humaneval_plus/gpt/"

In [151]:
_4o_mini_codes = []
for exp in os.listdir(os.path.join(gpt_results, 'gpt-4o-mini')):
    if exp!="codes":
        res_file = os.path.join(gpt_results, 'gpt-4o-mini', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 164
        _4o_mini_codes.append(data)
print(len(_4o_mini_codes))
_4o_mini_modelname = _4o_mini_codes[0][0]["result"][0]["model"]

_4o_mini_task_id = []
_4o_mini_prompt = []
_4o_mini_code = {}
for i in range(len(_4o_mini_codes)):
    codes = _4o_mini_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            _4o_mini_task_id.append(codes[j]["task_id"])
            _4o_mini_prompt.append(codes[j]["prompt"])
        c = extract_method_code_gpt(codes[j]["result"][0]["result"], _4o_mini_prompt[j])
        g_codes.append(c)
    _4o_mini_code[f"exp_{i}"] = g_codes
_4o_mini_code["task_id"] = _4o_mini_task_id
_4o_mini_code["prompt"] = _4o_mini_prompt

df = pd.DataFrame(_4o_mini_code)
df.to_csv(os.path.join("merged_humaneval_plus", f"{_4o_mini_modelname}.csv"))

10

In [6]:
_4o_codes = []
for exp in os.listdir(os.path.join(gpt_results, 'gpt-4o')):
    if exp!="codes":
        res_file = os.path.join(gpt_results, 'gpt-4o', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 164
        _4o_codes.append(data)
print(len(_4o_codes))
_4o_modelname = _4o_codes[0][0]["result"][0]["model"]
_4o_task_id = []
_4o_prompt = []
_4o_code = {}
for i in range(len(_4o_codes)):
    codes = _4o_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            _4o_task_id.append(codes[j]["task_id"])
            _4o_prompt.append(codes[j]["prompt"])
        c = extract_method_code_gpt(codes[j]["result"][0]["result"], _4o_prompt[j])
        g_codes.append(c)
    _4o_code[f"exp_{i}"] = g_codes
_4o_code["task_id"] = _4o_task_id
_4o_code["prompt"] = _4o_prompt

df = pd.DataFrame(_4o_code)
df.to_csv(os.path.join("merged_humaneval_plus", f"{_4o_modelname}.csv"))

10


In [14]:
_3turbo_codes = []
for exp in os.listdir(os.path.join(gpt_results, 'gpt-3.5-turbo')):
    if exp!="codes":
        res_file = os.path.join(gpt_results, 'gpt-3.5-turbo', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 164
        _3turbo_codes.append(data)
print(len(_3turbo_codes))
_3turbo_modelname = _3turbo_codes[0][0]["result"][0]["model"]
print(_3turbo_modelname)

_3turbo_task_id = []
_3turbo_prompt = []
_3turbo_code = {}
for i in range(len(_3turbo_codes)):
    codes = _3turbo_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            _3turbo_task_id.append(codes[j]["task_id"])
            _3turbo_prompt.append(codes[j]["prompt"])
        c =  _3turbo_prompt[j] + "\n"+  codes[j]["result"][0]["result"]
        c = extract_python_method_code(_3turbo_prompt[j], c)[0]
        g_codes.append(c)
    _3turbo_code[f"exp_{i}"] = g_codes
_3turbo_code["task_id"] = _3turbo_task_id
_3turbo_code["prompt"] = _3turbo_prompt

df = pd.DataFrame(_3turbo_code)
df.to_csv(os.path.join("merged_humaneval_plus", f"{_3turbo_modelname}.csv"))

10
gpt-3.5-turbo


# merge BigCodeBench


## deepseek-coder

In [3]:
deepseek_coder = "../results/bigcodebench/deepseek_coder"


In [5]:
base_codes = []
for exp in os.listdir(os.path.join(deepseek_coder, 'base')):
    if exp == "codes":
        pass
    else:
        res_file = os.path.join(deepseek_coder, 'base', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 1140
        base_codes.append(data)
len(base_codes)

10

In [6]:
base_modelname = base_codes[0][0]["result"][0]["model"].replace("/", "_")
# base_modelname

In [8]:
b_task_id = []
b_prompt = []
b_codes = {}
for i in range(len(base_codes)):
# for i in range(8):
    codes = base_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            b_task_id.append(codes[j]["task_id"])
            b_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    b_codes[f"exp_{i}"] = g_codes
b_codes["task_id"] = b_task_id
b_codes["prompt"] = b_prompt

df = pd.DataFrame(b_codes)
df.to_csv(os.path.join("merged_bigcodebench", f"{base_modelname}.csv"))
# df.to_csv(os.path.join("merged_bigcodebench", f"{base_modelname}_8.csv"))

In [4]:
merged_codes = []
for exp in os.listdir(os.path.join(deepseek_coder, 'merged')):
    if exp != "codes":
        res_file = os.path.join(deepseek_coder, 'merged', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 1140
        merged_codes.append(data)
len(merged_codes)

10

In [5]:
merged_modelname = merged_codes[0][0]["result"][0]["model"].replace("/", "_")


In [6]:
m_task_id = []
m_prompt = []
m_codes = {}
for i in range(len(merged_codes)):
    codes = merged_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            m_task_id.append(codes[j]["task_id"])
            m_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    m_codes[f"exp_{i}"] = g_codes
m_codes["task_id"] = m_task_id
m_codes["prompt"] = m_prompt

df = pd.DataFrame(m_codes)
df.to_csv(os.path.join("merged_bigcodebench", f"{merged_modelname}.csv"))


In [4]:
finetuned_codes = []
for exp in os.listdir(os.path.join(deepseek_coder, 'finetuned')):
    if exp != "codes":
        res_file = os.path.join(deepseek_coder, 'finetuned', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 1140
        finetuned_codes.append(data)
len(finetuned_codes)

10

In [5]:
finetuned_modelname = finetuned_codes[0][0]["result"][0]["model"].replace("/", "_")


In [6]:
f_task_id = []
f_prompt = []
f_codes = {}
for i in range(len(finetuned_codes)):
    codes = finetuned_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            f_task_id.append(codes[j]["task_id"])
            f_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    f_codes[f"exp_{i}"] = g_codes
f_codes["task_id"] = f_task_id
f_codes["prompt"] = f_prompt

df = pd.DataFrame(f_codes)
df.to_csv(os.path.join("merged_bigcodebench", f"{finetuned_modelname}.csv"))


## codellama

In [7]:
codellama = "../results/bigcodebench/codellama"


In [4]:
base_codes = []
for exp in os.listdir(os.path.join(codellama, 'base')):
    if exp!="codes":
        res_file = os.path.join(codellama, 'base', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 1140
        base_codes.append(data)
len(base_codes)

10

In [5]:
base_modelname = base_codes[0][0]["result"][0]["model"].replace("/", "_")
# base_modelname

In [6]:
b_task_id = []
b_prompt = []
b_codes = {}
base_modelname = base_codes[0][0]["result"][0]["model"].replace("/", "_")
for i in range(len(base_codes)):
    codes = base_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            b_task_id.append(codes[j]["task_id"])
            b_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    b_codes[f"exp_{i}"] = g_codes
b_codes["task_id"] = b_task_id
b_codes["prompt"] = b_prompt

df = pd.DataFrame(b_codes)
df.to_csv(os.path.join("merged_bigcodebench", f"{base_modelname}.csv"))

In [9]:
merged_codes = []
for exp in os.listdir(os.path.join(codellama, 'merged')):
    if exp!="codes":
        res_file = os.path.join(codellama, 'merged', exp, 'result.json')
        data = read_jsonl(res_file)
        assert (len(data)) == 1140
        merged_codes.append(data)
        
merged_modelname = merged_codes[0][0]["result"][0]["model"].replace("/", "_")
m_task_id = []
m_prompt = []
m_codes = {}
for i in range(len(merged_codes)):
    codes = merged_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            m_task_id.append(codes[j]["task_id"])
            m_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    m_codes[f"exp_{i}"] = g_codes
m_codes["task_id"] = m_task_id
m_codes["prompt"] = m_prompt

df = pd.DataFrame(m_codes)
df.to_csv(os.path.join("merged_bigcodebench", f"{merged_modelname}.csv"))

In [11]:
finetune_codes = []

for exp in os.listdir(os.path.join(codellama, 'finetuned')):
    if exp != "codes":
        res_file = os.path.join(codellama, 'finetuned', exp, 'result.json')
        data = read_jsonl(res_file)
        assert (len(data)) == 1140
        finetune_codes.append(data)
        
finetune_model_name = finetune_codes[0][0]["result"][0]["model"].replace("/", "_")

f_task_id = []
f_prompt = []
f_codes = {}
for i in range(len(finetune_codes)):
    codes = finetune_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            f_task_id.append(codes[j]["task_id"])
            f_prompt.append(codes[j]["prompt"])
        c, _, _ = extract_python_method_code(start_code=codes[j]["prompt"], generated_code=codes[j]["result"][0]["result"])
        g_codes.append(c)
    f_codes[f"exp_{i}"] = g_codes
f_codes["task_id"] = f_task_id
f_codes["prompt"] = f_prompt

df = pd.DataFrame(f_codes)
df.to_csv(os.path.join("merged_bigcodebench", f"{finetune_model_name}.csv"))


## GPT

In [6]:
gpt_results = "../results/bigcodebench/gpt/"

In [4]:
_4o_mini_codes = []
for exp in os.listdir(os.path.join(gpt_results, 'gpt-4o-mini')):
    if exp!="codes":
        res_file = os.path.join(gpt_results, 'gpt-4o-mini', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 1140
        _4o_mini_codes.append(data)
print(len(_4o_mini_codes))
_4o_mini_modelname = _4o_mini_codes[0][0]["result"][0]["model"]

_4o_mini_task_id = []
_4o_mini_prompt = []
_4o_mini_code = {}
for i in range(len(_4o_mini_codes)):
    codes = _4o_mini_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            _4o_mini_task_id.append(codes[j]["task_id"])
            _4o_mini_prompt.append(codes[j]["prompt"])
        c = extract_method_code_gpt(codes[j]["result"][0]["result"], _4o_mini_prompt[j])
        g_codes.append(c)
    _4o_mini_code[f"exp_{i}"] = g_codes
_4o_mini_code["task_id"] = _4o_mini_task_id
_4o_mini_code["prompt"] = _4o_mini_prompt

df = pd.DataFrame(_4o_mini_code)
df.to_csv(os.path.join("merged_bigcodebench", f"{_4o_mini_modelname}.csv"))
# df.to_csv(os.path.join("merged_bigcodebench", f"{_4o_mini_modelname}_1.csv"))

10


In [7]:
_4o_codes = []
for exp in os.listdir(os.path.join(gpt_results, 'gpt-4o')):
    if exp!="codes":
        res_file = os.path.join(gpt_results, 'gpt-4o', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 1140
        _4o_codes.append(data)
print(len(_4o_codes))
_4o_modelname = _4o_codes[0][0]["result"][0]["model"]
_4o_task_id = []
_4o_prompt = []
_4o_code = {}
for i in range(len(_4o_codes)):
    codes = _4o_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            _4o_task_id.append(codes[j]["task_id"])
            _4o_prompt.append(codes[j]["prompt"])
        c = extract_method_code_gpt(codes[j]["result"][0]["result"], _4o_prompt[j])
        g_codes.append(c)
    _4o_code[f"exp_{i}"] = g_codes
_4o_code["task_id"] = _4o_task_id
_4o_code["prompt"] = _4o_prompt

df = pd.DataFrame(_4o_code)
df.to_csv(os.path.join("merged_bigcodebench", f"{_4o_modelname}.csv"))
# df.to_csv(os.path.join("merged_bigcodebench", f"{_4o_modelname}_1.csv"))

10


In [5]:
_3turbo_codes = []
for exp in os.listdir(os.path.join(gpt_results, 'gpt-3.5-turbo')):
    if exp!="codes":
        res_file = os.path.join(gpt_results, 'gpt-3.5-turbo', exp, 'result.json')
        data = read_jsonl(res_file)
        assert len(data) == 1140
        _3turbo_codes.append(data)
print(len(_3turbo_codes))
_3turbo_modelname = _3turbo_codes[0][0]["result"][0]["model"]
print(_3turbo_modelname)

_3turbo_task_id = []
_3turbo_prompt = []
_3turbo_code = {}
for i in range(len(_3turbo_codes)):
    codes = _3turbo_codes[i]
    g_codes = []
    for j in range(len(codes)):  
        if i == 0: 
            _3turbo_task_id.append(codes[j]["task_id"])
            _3turbo_prompt.append(codes[j]["prompt"])
        c =  _3turbo_prompt[j] + "\n"+  codes[j]["result"][0]["result"]
        c = extract_python_method_code(_3turbo_prompt[j], c)[0]
        g_codes.append(c)
    _3turbo_code[f"exp_{i}"] = g_codes
_3turbo_code["task_id"] = _3turbo_task_id
_3turbo_code["prompt"] = _3turbo_prompt

df = pd.DataFrame(_3turbo_code)
df.to_csv(os.path.join("merged_bigcodebench", f"{_3turbo_modelname}.csv"))
# df.to_csv(os.path.join("merged_bigcodebench", f"{_3turbo_modelname}_1.csv"))

10
gpt-3.5-turbo
